# 01 · EDA temporal e integridad de los datos

Qué se responde aquí:

1. ¿Las dos fuentes se integran sin perder ni duplicar filas?
2. ¿Cómo cambian el fraude, el volumen y los faltantes a lo largo de 182 días?
3. ¿Las features de historial son realmente causales?

El notebook **no** recalcula nada pesado: lee el Parquet y los manifests que
produjo `fraud-adaptive data prepare`.

In [ ]:
import sys, json
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "configs").exists() and (ROOT.parent / "configs").exists():
    ROOT = ROOT.parent          # el notebook puede abrirse desde notebooks/
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from fraud_adaptive.pipeline import load_configs
from fraud_adaptive.tracking import read_json

CONFIGS = load_configs(ROOT / "configs")
RUN_DIR = ROOT / CONFIGS["base"]["paths"]["runs"] / "principal"
REPORTS = ROOT / CONFIGS["base"]["paths"]["reports"]

MANIFEST = read_json(ROOT / CONFIGS["base"]["paths"]["manifests"] / "sources.json")
DATA_SOURCE = MANIFEST.get("data_source", "desconocido")

if DATA_SOURCE == "sintetico_sustituto":
    print("AVISO: los datos son un SUSTITUTO SINTETICO. Ninguna cifra describe IEEE-CIS.")
else:
    print("Datos: IEEE-CIS Fraud Detection (particion train).")
print("Run:", RUN_DIR)

## 1. Auditoría del join

El número de filas debe ser invariante: un join many-to-many las multiplicaría en silencio.

In [ ]:
join = MANIFEST["join"]
esquema = MANIFEST["esquema"]

print(f"Transacciones : {esquema['n_transactions']:,} filas x {esquema['n_columns_transactions']} columnas")
print(f"Identidad     : {esquema['n_identity']:,} filas x {esquema['n_columns_identity']} columnas")
print(f"Prevalencia   : {esquema['prevalencia']:.4f}")
print()
for clave, valor in join.items():
    print(f"  {clave:<28} {valor}")

assert join["filas_invariantes"], "el join alteró el número de filas"
print("\nOK: el join es uno-a-uno y preserva todas las transacciones.")

## 2. Los dos relojes

`TransactionDT` es un **delta en segundos**, no una fecha. La hora derivada es relativa a un origen desconocido: sirve como ciclo de 24 h, no identifica hora local ni día laboral.

In [ ]:
eventos = pd.read_parquet(ROOT / CONFIGS["base"]["paths"]["processed"] / "eventos.parquet")

print(f"Eventos: {len(eventos):,}   columnas: {eventos.shape[1]}")
print(f"Días cubiertos: {eventos['dia'].min()} a {eventos['dia'].max()}")
print(f"Transacciones por día (mediana): {int(eventos.groupby('dia').size().median()):,}")
eventos[["TransactionDT", "s_rel", "dia", "semana", "hora"]].head()

## 3. Perfil semanal con intervalos de Wilson

Con prevalencias del ~3,5 % y miles de filas por semana, el intervalo de Wald puede salirse de [0,1]; Wilson no.

In [ ]:
from fraud_adaptive.data import weekly_profile

semanal = weekly_profile(eventos, target="isFraud", amount_col="TransactionAmt")
semanal[["semana", "n", "fraudes", "prevalencia", "wilson_low", "wilson_high",
         "cobertura_identidad"]].round(4).head(12)

## 4. Faltantes por familia

IEEE-CIS tiene cientos de columnas anónimas. La familia (V, C, D, M, id_, card, addr) es la unidad interpretable.

In [ ]:
from fraud_adaptive.data import missingness_by_family

missingness_by_family(eventos).round(3)

## 5. Anomalías antes de culpar al drift

Un hueco de captura o un pico de duplicados explican una alerta mejor que un cambio de comportamiento. Se descartan primero.

In [ ]:
from fraud_adaptive.data import detect_anomalies

print(json.dumps(detect_anomalies(eventos, amount_col="TransactionAmt", target="isFraud"),
                 indent=2, ensure_ascii=False))

## 6. Calidad de los proxies de entidad

Un proxy agrupa comportamiento; **no identifica a una persona**. Se mide en vez de asumirse.

In [ ]:
for nombre, calidad in MANIFEST["features"]["proxies"].items():
    print(f"{nombre}:")
    for clave, valor in calidad.items():
        if clave != "nota":
            print(f"    {clave:<30} {valor}")
    print()

## 7. Causalidad: la prueba que importa

Las features se emiten **antes** de actualizar el estado. La comprobación decisiva
es que añadir un evento futuro no mueva ninguna fila anterior.

In [ ]:
from fraud_adaptive.features import prepare_features

base = pd.DataFrame({
    "TransactionID": [1, 2, 3], "TransactionDT": [100, 200, 300],
    "TransactionAmt": [10.0, 20.0, 30.0],
    "card1": ["A"]*3, "card2": [1]*3, "card3": [1]*3, "card4": ["visa"]*3,
    "card5": [1]*3, "card6": ["debit"]*3, "addr1": [1]*3,
    "DeviceType": ["desktop"]*3, "DeviceInfo": ["Windows"]*3,
    "has_identity": [1]*3, "isFraud": [0]*3,
})
con_futuro = pd.concat([base, base.iloc[[0]].assign(
    TransactionID=4, TransactionDT=400, TransactionAmt=999999.0)], ignore_index=True)

a, _ = prepare_features(base)
b, _ = prepare_features(con_futuro)
# Solo las features numéricas de historial: `card_proxy` es la clave de entidad, texto.
cols = [c for c in a.columns
        if c.startswith("card_") and c != "card_proxy" and pd.api.types.is_numeric_dtype(a[c])]

iguales = np.allclose(a[cols].fillna(-1).to_numpy(), b[cols].head(3).fillna(-1).to_numpy())
print("Un evento futuro de monto extremo no altera las filas anteriores:", iguales)
assert iguales
a[["TransactionID", "card_cnt_expansivo", "card_cnt_24h", "card_amt_rezago"]]

## 8. Figura de EDA

Generada por `fraud-adaptive report build`.

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(REPORTS / "figures" / "eda_temporal.png")))